# Focus

This notebooks focus is on checking the data and cleaning data from wikipedia

# Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Make sure data and processed directories exist

In [ ]:
RAW_DATA_PATH = Path("../data/raw")
PROCESSED_DATA_PATH = Path("../data/processed")

In [ ]:
if not RAW_DATA_PATH.exists():
    print(f"Creating directory: {RAW_DATA_PATH}")
    RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)
else:
    print(f"Directory already exists: {RAW_DATA_PATH}")
if not PROCESSED_DATA_PATH.exists():
    print(f"Creating directory: {PROCESSED_DATA_PATH}")
    PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
else:
    print(f"Directory already exists: {PROCESSED_DATA_PATH}")

# Load Raw Data

First, I loaded the file normally to inspect its structure and check what needs cleaning.

In [ ]:
sp500 = pd.read_csv(RAW_DATA_PATH / "sp500_constituents.csv")
sp500.head()

In [ ]:
sp500.info()

# Check for Missing Values and Duplicates

In [ ]:
sp500.isnull().sum()

In [ ]:
sp500.duplicated(subset='ticker').sum()

No missing values and no duplicate tickers, so the raw table is already in good shape structurally. The cleaning that actually matters here is fixing up the ticker symbol formatting so it can be used to merge with the price data later.

# Fix Ticker Symbol Formatting

Wikipedia writes some tickers with a period, like BRK.B, while Yahoo Finance and most other sources use a dash instead, like BRK-B. If this isn't fixed, any future merge between this table and price data will silently drop those rows.

In [ ]:
dotted_tickers = sp500[sp500['ticker'].str.contains('\\.', regex=True)]
dotted_tickers

This shows which tickers actually have a period in them, so we know exactly what's getting changed instead of blindly replacing everything.

In [ ]:
sp500['ticker'] = sp500['ticker'].str.replace('.', '-', regex=False)

In [ ]:
sp500[sp500['ticker'].str.contains('-', regex=False)]

Confirmed the tickers that had a period now show up with a dash instead.

# Clean Whitespace and Text Formatting

In [ ]:
sp500['ticker'] = sp500['ticker'].str.strip()
sp500['company_name'] = sp500['company_name'].str.strip()
sp500['gics_sector'] = sp500['gics_sector'].str.strip()

Stripping any extra whitespace just in case, this is a quick safety check since stray spaces can cause silent mismatches during a merge later.

# Inspect Cleaned Data

In [ ]:
sp500.head()

In [ ]:
sp500['gics_sector'].nunique()

Confirms the number of unique sectors, which is a quick sanity check that the sector column is consistent and clean.

# Save Cleaned Data

In [ ]:
sp500.to_csv(PROCESSED_DATA_PATH / "sp500_constituents_clean.csv", index=False)

# Final Summary

In this notebook it was focused on loading in the raw S&P 500 wikipedia table, checking for missing values and duplicates, fixing ticker symbol formatting so it can match up with price data later, cleaning up whitespace, and saving the processed dataset.

### Files Created
- `data/processed/sp500_constituents_clean.csv`